<img src="https://raw.githubusercontent.com/OmarLi-360/excel-corporativo-12-claves/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Clave 10 · Fórmulas de matriz dinámica
**Excel Corporativo · 12 Claves** — FILTRAR, UNICOS, ORDENAR, TOMAR y LET · ⏱ 25 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/10_matriz_dinamica.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-10)

<sub>Si al abrir un enlace Google muestra «Redirect Notice», haz clic en el enlace azul de esa página: es un aviso de seguridad de Google, no un error.</sub>

**Al terminar podrás:**
1. Ver la lista que devuelve la fórmula del top de clientes para cualquier sede.
2. Medir cuántos clientes concentran la mitad de la venta de cada sede.
3. Contar clientes únicos por sede y año, como hace CONTARA(UNICOS(…)).

---
### Cómo usar este cuaderno (no necesitas programar)
1. **Guarda tu copia:** *Archivo › Guardar una copia en Drive*. Así tus cambios no se pierden. Si no lo haces, no pasa nada grave: el original está protegido y nadie puede dañarlo desde aquí, pero lo que cambies se borra al cerrar.
2. **Ejecuta todo:** *Entorno de ejecución › Ejecutar todas* (Ctrl+F9). Si Colab avisa que el cuaderno no es de Google, elige *Ejecutar de todos modos*.
3. **Juega con los controles:** las celdas con listas desplegables a la derecha se pueden cambiar. Elige una opción y pulsa el botón ▶ de esa celda para redibujar.

El código está oculto a propósito: aquí importa entender el dato y el gráfico. Si tienes curiosidad, haz doble clic en una celda para verlo.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/excel-corporativo-12-claves"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

## 1 · La lista que se actualiza sola
En la hoja **Top_Clientes** de `Andina_listo_para_dinamicas.xlsx`, una sola fórmula devuelve los mejores clientes de la sede elegida:

`=LET(c;D6#;v;E6#;TOMAR(ORDENAR(APILARH(c;v);2;-1);$B$4))`

donde D6# es la lista de clientes únicos (`UNICOS(FILTRAR(…))`) y E6# su venta (`SUMAR.SI.CONJUNTO(…;D6#;…)`). Aquí ves el mismo resultado. Cambia la sede, el año o el número de clientes.

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Quiénes son los mejores clientes de la sede y cuánto pesan? |
> | **Qué se grafica** | Barras horizontales ordenadas: venta neta de los primeros clientes, en millones |
> | **Por qué esta forma** | Es un ranking: las barras ordenadas se leen de arriba abajo, como la lista de Excel. |
> | **Qué se descartó** | Una tabla sola: muestra los nombres, pero no la distancia entre el primero y los demás. |
> | **Cómo se lee** | La primera barra es el cliente más importante; el título dice cuánto suman todos juntos. |
> | **En Excel 365** | Hoja Top_Clientes: sede en B2, año en B3 y cuántos en B4. |

In [ ]:
# @title Gráfico 1 · Top de clientes de la sede
sede = "Bogotá"  # @param ["Bogotá", "Medellín", "Cali", "Barranquilla", "Bucaramanga", "Pereira"]
anio = 2025  # @param [2024, 2025]
cuantos = 10  # @param {type:"slider", min:5, max:20, step:1}
por_cliente = (validas[(validas.ciudad == sede) & (validas.anio == anio)]
               .groupby("razon_social")["subtotal"].sum().sort_values(ascending=False))
top = por_cliente.head(cuantos)[::-1] / 1e6
fig = go.Figure(go.Bar(x=top.values, y=top.index, orientation="h", marker_color=[GRIS] * (len(top) - 1) + [AZUL],
                       text=[f"{v:,.0f}".replace(",", ".") for v in top.values], textposition="outside", cliponaxis=False))
fig.update_layout(title=f"Los {cuantos} primeros clientes de {sede} suman el {porc(por_cliente.head(cuantos).sum() / por_cliente.sum())} de su venta {anio}",
                  xaxis_title=MM, height=200 + 26 * cuantos, margin=dict(l=10))
dar_aire(fig, top.max())
fig.show()
display(Markdown(f"UNICOS devolvió **{len(por_cliente)} clientes** en {sede} durante {anio}; TOMAR se quedó con {cuantos}."))

## 2 · ¿Cuántos clientes hacen la mitad de la venta?

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué tan concentrada está la venta de cada sede? |
> | **Qué se grafica** | Líneas: porcentaje acumulado de la venta a medida que se suman clientes, de mayor a menor · Línea punteada: 50 % |
> | **Por qué esta forma** | Una curva por sede permite comparar la concentración: la que sube más rápido depende de menos clientes. |
> | **Qué se descartó** | Barras del top 10 por sede: seis gráficos que no se comparan entre sí. |
> | **Cómo se lee** | Donde cada curva cruza el 50 % está el número de clientes que hacen la mitad de la venta. |
> | **En Excel 365** | Con la lista de A7 ordenada: =SUMA(TOMAR(B7#;n))/SUMA(B7#) para distintos n, o una columna de acumulado con =ESCANEAR. |

In [ ]:
# @title Gráfico 2 · Concentración de la venta por sede
fig = go.Figure()
cruces = {}
for i, s in enumerate(ORDEN_CIUDADES):
    x = validas[(validas.ciudad == s) & (validas.anio == anio)].groupby("razon_social")["subtotal"].sum().sort_values(ascending=False)
    acum = (x.cumsum() / x.sum()).values
    cruces[s] = int((acum < 0.5).sum() + 1)
    fig.add_scatter(x=list(range(1, len(acum) + 1)), y=acum, name=f"{s} ({cruces[s]})", mode="lines",
                    line=dict(width=3 if s == sede else 1.5, color=AZUL if s == sede else GRIS))
fig.add_hline(y=0.5, line_dash="dot", line_color=MAGENTA, annotation_text="50 %", annotation_position="bottom right")
fig.update_layout(title=f"{anio}: en {sede} bastan {cruces[sede]} clientes para la mitad de la venta",
                  xaxis_title="Clientes, de mayor a menor venta", yaxis_tickformat=".0%", height=440, legend=dict(title="Clientes para el 50 %:", orientation="h", y=-0.25))
fig.show()

## 3 · Clientes únicos

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cuántos clientes distintos compraron en cada sede? |
> | **Qué se grafica** | Columnas agrupadas: clientes únicos por sede en 2024 y 2025 |
> | **Por qué esta forma** | Comparar dos años por sede: columnas lado a lado, con el año más reciente en el color fuerte. |
> | **Qué se descartó** | Líneas: con solo dos puntos por sede no hay tendencia que mostrar. |
> | **Cómo se lee** | Si una sede vende más con menos clientes, su venta depende más de pocos clientes. |
> | **En Excel 365** | =CONTARA(UNICOS(FILTRAR(tVentas[cliente];(tVentas[ciudad]=A2)*(tVentas[anio]=2025)))) |

In [ ]:
# @title Gráfico 3 · Clientes únicos por sede y año
unicos = validas.groupby(["ciudad", "anio"])["id_cliente"].nunique().unstack().reindex(ORDEN_CIUDADES)
fig = go.Figure()
for a, color in [(2024, GRIS), (2025, AZUL)]:
    fig.add_bar(name=str(a), x=unicos.index, y=unicos[a], marker_color=color, text=unicos[a], textposition="outside")
cambio = (unicos[2025] - unicos[2024])
fig.update_layout(barmode="group", yaxis_title="Clientes únicos", height=380, legend_title="",
                  title=f"De 2024 a 2025 la sede que más clientes sumó fue {cambio.idxmax()} (+{cambio.max()})")
fig.show()

## 4 · Retos de interpretación

In [ ]:
# @title Reto 1 · ¿Quién es el mejor cliente de Bogotá en 2025?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Laboratorios Orquídea S.A.S.", "Alcaldía de Chicamocha", "Transportes El Prado S.A.S.", "Laboratorios Santa Fe S.A."]
revisar(respuesta, validas[(validas.ciudad == "Bogotá") & (validas.anio == 2025)].groupby("razon_social")["subtotal"].sum().idxmax(), """Es la primera fila de la lista que devuelve la fórmula con B2 = Bogotá y B3 = 2025.""")

In [ ]:
# @title Reto 2 · ¿Cuántos clientes hacen la mitad de la venta de Bogotá en 2025?
respuesta = "Elige una opción"  # @param ["Elige una opción", "10", "15", "22", "45"]
revisar(respuesta, str(int(((lambda x: x.cumsum() / x.sum())(validas[(validas.ciudad == "Bogotá") & (validas.anio == 2025)].groupby("razon_social")["subtotal"].sum().sort_values(ascending=False)) < 0.5).sum() + 1)), """Es el punto donde la curva de Bogotá cruza la línea del 50 % (con año 2025 en el gráfico 1).""")

In [ ]:
# @title Reto 3 · La fórmula del top muestra #¡DESBORDAMIENTO!. ¿Qué significa?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Que la fórmula está mal escrita", "Que hay celdas ocupadas donde debe derramarse el resultado", "Que faltan datos", "Que el libro es muy grande"]
revisar(respuesta, "Que hay celdas ocupadas donde debe derramarse el resultado", """Se despeja el espacio y el resultado aparece; la fórmula no se toca.""")

## Apóyate en IA
Para construir una lista que se actualice sola:

```text
En Excel 365 en español (separador ;) tengo la Tabla tVentas con cliente, ciudad, anio, estado y subtotal.
Quiero los 10 clientes con más venta neta de la sede de B2 y el año de B3, sin anuladas, en una sola fórmula con LET.
Explica cada paso y qué hago si aparece #¡DESBORDAMIENTO!.
```

> **Verifica siempre.** Compara la respuesta de la IA con un dato que ya conozcas. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* Una fórmula de matriz dinámica devuelve una lista completa que se recalcula sola.
* D6# se refiere a todo el resultado derramado, con el tamaño que tenga.
* LET hace legible una fórmula larga dando nombre a cada paso.

**Siguiente:** [Clave 11 · Gráficos y narrativa](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-11) · [Cuaderno 11](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/11_graficos_y_narrativa.ipynb)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>